<a href="https://colab.research.google.com/github/Mhmd-Seyed/ecommerce-behavior-analytics/blob/main/03_spark_multidimensional_analysis_ipynb_adl%C4%B1_not_defterinin_kopyas%C4%B1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import glob
from google.colab import drive
from pyspark.sql import SparkSession

# 1. Google Drive'ı bağla
drive.mount('/content/drive')

# 2. Spark oturumunu başlat
spark = SparkSession.builder \
    .appName("KategoriAnalizSorgusu") \
    .getOrCreate()

# 3. events_clean.parquet dosyasının Drive'daki tam yolunu otomatik bul
dosyalar = glob.glob("/content/drive/MyDrive/**/events_clean.parquet", recursive=True)
if not dosyalar:
    raise FileNotFoundError("events_clean.parquet dosyası Google Drive içinde bulunamadı!")

parquet_yolu = dosyalar[0]
print("Kullanılan dosya:", parquet_yolu)

# 4. Veriyi oku ve SQL tablosu oluştur
events_df = spark.read.parquet(parquet_yolu)
events_df.createOrReplaceTempView("events")

# 5. Kategori bazlı kişi, etkileşim, sipariş ve ciro sorgusu
query = """
SELECT
    COALESCE(NULLIF(SPLIT(category_code, '\\\\.')[0], ''), 'unknown') AS ana_kategori,
    COUNT(DISTINCT user_id) AS benzersiz_kisi_sayisi,
    COUNT(DISTINCT product_id) AS urun_sayisi,
    SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS goruntuleme_sayisi,
    SUM(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) AS sepete_ekleme_sayisi,
    SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS siparis_tamamlama,
    ROUND(SUM(CASE WHEN event_type = 'purchase' THEN price ELSE 0 END), 2) AS toplam_ciro,
    ROUND(
        SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) * 100.0 /
        NULLIF(SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END), 0),
        2
    ) AS donusum_orani_yuzde
FROM events
GROUP BY 1
ORDER BY toplam_ciro DESC
"""

spark.sql(query).show(20, truncate=False)

In [ ]:
import glob
import pandas as pd
from google.colab import drive

# 1. Google Drive'ı bağla
drive.mount('/content/drive')

# 2. category_analysis.csv dosyasını Drive'da otomatik bul ve oku
dosyalar = glob.glob("/content/drive/MyDrive/**/category_analysis.csv", recursive=True)
if not dosyalar:
    raise FileNotFoundError("category_analysis.csv dosyası Drive içinde bulunamadı!")

c_df = pd.read_csv(dosyalar[0])

# 3. Tüm kategorilerin sayısal tablosunu göster
print("--- TÜM KATEGORİLER ---")
display(c_df)

# 4. Sadece kids ve apparel satırlarını filtrele
print("--- KIDS VE APPAREL DETAYI ---")
display(c_df[c_df["main_category"].isin(["kids", "apparel"])])